# CA1: Adult Income – Data Preparation and EDA
**Domain:** Population and Society  
**Dataset:** UCI Adult (Census Income), Becker and Kohavi (1996)  
**Source:** https://archive.ics.uci.edu/dataset/2/adult

- **Part A:** motivation, source and reference, and what each column means
- **Part B (Task 1):** data preparation: size, attributes, missing values, duplicates, outliers, feature engineering and challenges
- **Part C (Task 2):** exploratory data analysis: variation, covariation, four questions, unusual patterns and findings

# Part A: Problem Domain and Dataset

## A1. Problem domain and motivation
I chose the **Population and Society** domain because income inequality affects almost every country, including Pakistan. People often assume that education and hard work decide how much someone earns, but factors such as gender, race, marital status and type of job may also play a large part. I wanted to test these ideas with real data rather than opinion.

The **UCI Adult (Census Income)** dataset suits this goal well. It contains real records from the 1994 US Census for about 48,000 people, with a clear outcome (whether a person earns more than $50,000 a year) and a mix of personal and work details. It also has missing values and outliers, so it gives a realistic chance to practise data cleaning, and it has both text and numeric columns for encoding and PCA.

**Problem to explore:** Which demographic and employment factors are most strongly linked to earning more than $50K a year, and are there clear income gaps by gender, race or education level?

## A2. Source and Harvard reference
**Link:** https://archive.ics.uci.edu/dataset/2/adult

**Harvard reference:**  
Becker, B. and Kohavi, R. (1996) *Adult* [Dataset]. UCI Machine Learning Repository. Available at: https://doi.org/10.24432/C5XW20 (Accessed: 17 November 2026).

In-text: (Becker and Kohavi, 1996)

## A3. What each column means

| Column | Type | Meaning |
|---|---|---|
| age | Numeric | Age of the person in years |
| workclass | Categorical | Type of employer (private, self-employed, government, etc.) |
| fnlwgt | Numeric | **Census sampling weight**: roughly how many people in the US population this record represents. It describes the sample, **not a characteristic of the person**, so it is usually left out of analysis and PCA. |
| education | Categorical | Highest education level as text (e.g. Bachelors, HS-grad) |
| education_num | Numeric | The same education level as a number (1 = Preschool … 16 = Doctorate). **It holds the same information as `education`**, so only one of the two should be used in modelling and PCA. |
| marital_status | Categorical | Marital status (married, never married, divorced, etc.) |
| occupation | Categorical | Type of job (e.g. Tech-support, Sales, Craft-repair) |
| relationship | Categorical | Role in the household (husband, wife, own-child, etc.) |
| race | Categorical | Race recorded in the census |
| sex | Categorical | Male or Female |
| capital_gain | Numeric | Income from investments (e.g. selling shares), in US dollars |
| capital_loss | Numeric | Losses from investments, in US dollars |
| hours_per_week | Numeric | Hours worked per week |
| native_country | Categorical | Country of birth |
| income | Categorical (target) | Whether yearly income is `<=50K` or `>50K` US dollars |

# Part B: Data Preparation (Task 1)

## B0. Load the raw data

In [ ]:
# --- Import libraries ---
import io                         # treat downloaded bytes like a file
import zipfile                    # open the .zip file from UCI
import urllib.request             # download from the web
import numpy as np                # numeric functions
import pandas as pd               # tables (DataFrames)
import matplotlib.pyplot as plt   # charts

URL = "https://archive.ics.uci.edu/static/public/2/adult.zip"
COLUMNS = ["age", "workclass", "fnlwgt", "education", "education_num",
           "marital_status", "occupation", "relationship", "race", "sex",
           "capital_gain", "capital_loss", "hours_per_week", "native_country",
           "income"]

# Download the zip file into memory
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read()))

def read_part(name, skip, na=None):
    """Read one file from the zip.
    skip = junk lines at the top (adult.test has 1).
    na   = which text should count as a missing value (None = nothing)."""
    return pd.read_csv(io.TextIOWrapper(z.open(name)), header=None, names=COLUMNS,
                       skiprows=skip, skipinitialspace=True, na_values=na)

# Read the train and test files EXACTLY as they are (no cleaning yet),
# so we can show the problems in the raw data
raw = pd.concat([read_part("adult.data", 0), read_part("adult.test", 1)],
                ignore_index=True)
# Save a copy of the original data (only the income full stop fixed)
raw_out = raw.copy()
raw_out["income"] = raw_out["income"].str.rstrip(".")
raw_out.to_csv("adult_raw.csv", index=False)
print("Saved adult_raw.csv")
raw.head()

## B1. Size before cleaning
`.shape` gives (rows, columns). The train and test files together hold the full dataset.

In [ ]:
# Size of the raw dataset
print("Raw shape (rows, columns):", raw.shape)

## B2. Attributes (columns)
### B2.1 Column types
`.info()` shows each column's data type. Text columns (`object`) are **categorical**; integer columns are **numeric**.

In [ ]:
# Data type of every column and the number of non-empty values
raw.info()

In [ ]:
# Number of different values in each column
# (helps decide how categorical columns will be encoded later in Task 3)
raw.nunique().sort_values()

In [ ]:
# Summary statistics for the numeric columns (min, max, mean, quartiles)
raw.describe().round(1)

### B2.2 Columns that will be removed (and why)
**`education` and `education_num` hold the same information.** `education` is the level as text (e.g. *Bachelors*) and `education_num` is the same level as a number (e.g. 13). The code below proves that each text level maps to exactly one number. We keep `education_num`, because it is already numeric and keeps the natural order of levels (Preschool < … < Doctorate), and remove `education`.

**`fnlwgt` is a census sampling weight**, not a feature of the person. It tells us how many people in the US population a record represents. The code below shows it has almost no link with income.

In [ ]:
# Each education text level should map to ONE education_num value
mapping = raw.groupby("education")["education_num"].unique().sort_values()
print(mapping)
print("\nEvery level maps to exactly one number:",
      raw.groupby("education")["education_num"].nunique().eq(1).all())

In [ ]:
# Correlation between fnlwgt and income (1 = >50K, 0 = <=50K).
# A value close to 0 means fnlwgt tells us almost nothing about income.
income_01 = raw["income"].str.contains(">50K").astype(int)
print("Correlation of fnlwgt with income:", round(raw["fnlwgt"].corr(income_01), 4))

*These two columns are removed in section B4.2, after duplicates are checked (the order matters: see challenge 4).*

## B3. Missing values
### B3.1 Finding hidden missing values
In the raw files, missing values are written as **`?`**, not left empty. Pandas therefore reads them as normal text, and `.isna()` reports **no missing values at all**, even though data is missing.

In [ ]:
# Pandas thinks nothing is missing...
print("Missing values found by isna():", raw.isna().sum().sum())

# ...but the '?' symbol appears in several columns
print("\nCount of '?' in each column:")
q = (raw == "?").sum()
print(q[q > 0])

### B3.2 Fixing the labels and the '?' values
Two fixes are applied to the raw data:
- **`?` → NaN**, so pandas recognises the values as missing.
- **Income labels:** the test file writes `>50K.` and `<=50K.` with a full stop, so the target column has **4 labels instead of 2**. The full stop is removed.

In [ ]:
# The income column has 4 labels before the fix
print("Income labels BEFORE:", raw["income"].unique())

df = raw.copy()                              # work on a copy; keep raw unchanged
df = df.replace("?", np.nan)                 # mark '?' as a real missing value
df["income"] = df["income"].str.rstrip(".")  # '>50K.' -> '>50K'

print("Income labels AFTER: ", df["income"].unique())

### B3.3 How much is missing?

In [ ]:
# Missing values per column, as a count and as a percentage of all rows
missing = df.isna().sum()
missing = missing[missing > 0]
print(pd.DataFrame({"missing": missing,
                    "percent": (missing / len(df) * 100).round(2)}))

rows_missing = df.isna().any(axis=1).sum()   # rows with at least one gap
print(f"\nRows with at least one missing value: {rows_missing:,} "
      f"({rows_missing / len(df) * 100:.2f}% of all rows)")

### B3.4 Are the values missing at random?
Before choosing a method, we check *where* the gaps are. If `workclass` and `occupation` are usually missing together, the gap is probably about people who were not working, not random errors.

In [ ]:
# How often workclass and occupation are missing together
print(pd.crosstab(df["workclass"].isna(), df["occupation"].isna(),
                  rownames=["workclass missing"], colnames=["occupation missing"]))

# Rows where occupation is missing but workclass is NOT: what is their workclass?
print("\nWorkclass when only occupation is missing:")
print(df.loc[df["occupation"].isna() & df["workclass"].notna(), "workclass"].value_counts())

In [ ]:
# Does the share of high earners differ between rows with and without gaps?
has_gap = df.isna().any(axis=1)
print("Share earning >50K")
print("  rows WITH missing values:   ", round((df.loc[has_gap,  "income"] == ">50K").mean(), 3))
print("  rows WITHOUT missing values:", round((df.loc[~has_gap, "income"] == ">50K").mean(), 3))

### B3.5 Handling the missing values: `dropna()`
**Method chosen: remove rows with missing values (`dropna()`).**

**Why:**
- The missing values are all in **categorical** columns, so mean or median filling is not possible.
- Only about **7% of rows** are affected, so more than 45,000 rows remain, far above the 3,000 minimum.
- Filling with `"Unknown"` would add an artificial category to three columns. That category would be mixed with real ones during encoding and PCA in Task 3, and it has no clear meaning.
- Filling with the **mode** (most common value) would wrongly label people who were not working as, for example, *Private* employees.

**Trade-off:** section B3.4 shows the gaps are **not completely random**. `workclass` and `occupation` are mostly missing together, and the rows with gaps contain fewer high earners. Dropping them may slightly under-represent people outside regular work. Because the loss is small and the dataset stays large, this is accepted and noted as a limitation.

In [ ]:
before = len(df)
df = df.dropna()                     # remove every row that has a missing value
print(f"Rows removed: {before - len(df):,}")
print("Missing values left:", df.isna().sum().sum())
print("Shape now:", df.shape)

## B4. Duplicates and redundant columns
### B4.1 Removing duplicate rows
Exact duplicate rows add no information and would give some people extra weight in the analysis. They are checked **before** `fnlwgt` is removed (see challenge 4).

In [ ]:
# Count rows that are exact copies of an earlier row (all 15 columns equal)
n_dup = df.duplicated().sum()
print("Exact duplicate rows:", n_dup)

# Show a few of them
df[df.duplicated(keep=False)].sort_values(COLUMNS).head(6)

In [ ]:
before = len(df)
df = df.drop_duplicates()            # keep the first copy, remove the rest
print(f"Duplicate rows removed: {before - len(df):,}")
dup_left = df.duplicated().sum()     # should be 0; saved for the summary table
print("Exact duplicates left:", dup_left)
print("Shape now:", df.shape)

### B4.2 Removing `fnlwgt` and `education`

In [ ]:
# Why the order matters: WITHOUT fnlwgt, many different people look identical
print("Rows that would look like duplicates if fnlwgt were removed first:",
      df.drop(columns=["fnlwgt", "education"]).duplicated().sum())

# Remove the two columns explained in section B2.2
df = df.drop(columns=["fnlwgt", "education"])
print("\nShape after removing columns:", df.shape)
print("Remaining columns:", df.columns.tolist())

## B5. Out-of-range data and outliers
### B5.1 Out-of-range (invalid) values
First, check that every value is **possible**. For example, an age must be within a sensible working range and hours per week cannot be more than 168.

In [ ]:
# Sensible limits for each numeric column
limits = {"age": (17, 100), "education_num": (1, 16),
          "capital_gain": (0, 100000), "capital_loss": (0, 100000),
          "hours_per_week": (1, 168)}

# Count values outside the limits
for col, (lo, hi) in limits.items():
    bad = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f"{col:15s} allowed {lo}-{hi:<7}  min={df[col].min():<6} max={df[col].max():<6} invalid={bad}")

### B5.2 Outliers with the IQR method
A value is an outlier if it is more than 1.5 × IQR below Q1 or above Q3.

In [ ]:
num_cols = df.select_dtypes(include="number").columns   # numeric columns only
rows = []
for c in num_cols:
    q1, q3 = df[c].quantile([0.25, 0.75])            # 1st and 3rd quartiles
    iqr = q3 - q1                                    # interquartile range
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr          # outlier limits
    n = ((df[c] < lo) | (df[c] > hi)).sum()          # count outside limits
    rows.append([c, q1, q3, iqr, n, round(n / len(df) * 100, 2)])

iqr_table = pd.DataFrame(rows, columns=["column", "Q1", "Q3", "IQR", "outliers", "% of rows"])
iqr_table

In [ ]:
# Box plots: dots beyond the whiskers are IQR outliers
df[num_cols].plot(kind="box", subplots=True, layout=(1, len(num_cols)),
                  figsize=(14, 4), sharey=False)
plt.suptitle("Numeric columns after cleaning: outliers")
plt.tight_layout(); plt.show()

### B5.3 Checking the suspicious values
Some extreme values look like **caps**, where the census recorded one top value for everyone above it, rather than errors.

In [ ]:
# capital_gain: how many zeros, and how many exactly 99,999?
print(f"capital_gain = 0:      {(df['capital_gain'] == 0).mean() * 100:.1f}% of rows")
print(f"capital_gain = 99,999: {(df['capital_gain'] == 99999).sum()} rows")
print("Next-highest capital_gain after 99,999:",
      df.loc[df["capital_gain"] < 99999, "capital_gain"].max())

# hours_per_week = 99 and age = 90 (the highest values in each column)
print(f"\nhours_per_week = 99: {(df['hours_per_week'] == 99).sum()} rows")
print(f"age = 90:            {(df['age'] == 90).sum()} rows")

# What share of people with capital_gain = 99,999 earn >50K?
print("\nShare earning >50K when capital_gain = 99,999:",
      round((df.loc[df['capital_gain'] == 99999, 'income'] == ">50K").mean(), 3))

### B5.4 Decision: keep the outliers
**All outliers are kept. No rows are removed or capped.**

**Why:**
- Section B5.1 found **no invalid values**. Every value is possible, so these are real but rare observations, not errors.
- **`capital_gain`:** most values are 0, so Q1 = Q3 = 0 and the IQR is 0. The IQR rule then marks *every* non-zero value as an outlier. Removing them would delete almost everyone with investment income, and those people are mostly high earners, which is exactly the group we study. The value **99,999 is a census cap** (there is a large gap below it), so it means "very high gain", not a typing error.
- **`hours_per_week = 99`** and **`age = 90`** are the highest values the census recorded (caps). Some people really do work very long hours, and people aged 90 exist.
- **Capping** (for example with `.clip()`) would hide real differences between people with high incomes.

To reduce the effect of the very skewed capital columns in later tasks, section B6 adds a simple yes/no feature. PCA in Task 3 will also use **scaled** values.

## B6. Feature engineering
New columns are created to make the data easier to analyse and model:
1. **`income` → 0/1**: models and correlation need a numeric target (1 = earns >50K).
2. **`age_group`**: groups ages into ranges with `pd.cut()`, which are easier to compare in charts than 70+ single ages.
3. **`has_capital_gain`**: 1 if the person had any investment income. This keeps the useful signal from the very skewed `capital_gain` column in a simple form.

In [ ]:
# 1. Income as 0/1
df["income"] = (df["income"] == ">50K").astype(int)
print("Income value counts (0 = <=50K, 1 = >50K):")
print(df["income"].value_counts())
print(f"Share of >50K: {df['income'].mean() * 100:.1f}%  <- the classes are imbalanced")

In [ ]:
# 2. Age groups (bins are 'right-closed': (16, 25] means 17-25)
bins   = [16, 25, 35, 45, 55, 65, 100]
labels = ["17-25", "26-35", "36-45", "46-55", "56-65", "66+"]
df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels)

# 3. Any capital gain? (1 = yes, 0 = no)
df["has_capital_gain"] = (df["capital_gain"] > 0).astype(int)

# Check: share of high earners in each age group and by capital gain
print(df.groupby("age_group", observed=True)["income"].mean().round(3))
print()
print(df.groupby("has_capital_gain")["income"].mean().round(3))

In [ ]:
# Bar chart: the new age_group feature clearly separates income levels
df.groupby("age_group", observed=True)["income"].mean().mul(100).plot(kind="bar", color="steelblue")
plt.ylabel("% earning >50K"); plt.xlabel("Age group")
plt.title("Share of high earners by age group")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## B7. Before/after summary table

In [ ]:
from IPython.display import Markdown, display

raw_fixed = raw.replace("?", np.nan)                     # '?' counted as missing
raw_fixed["income"] = raw_fixed["income"].str.rstrip(".")  # same labels, so duplicates are compared fairly
summary = pd.DataFrame({
    "Before cleaning": [f"{raw.shape[0]:,}", raw.shape[1],
                        f"{raw_fixed.isna().sum().sum():,}",
                        f"{raw_fixed.isna().any(axis=1).sum():,}",
                        f"{raw_fixed.duplicated().sum():,}",
                        raw["income"].nunique()],
    "After cleaning": [f"{df.shape[0]:,}", df.shape[1],
                       df.isna().sum().sum(), 0,
                       dup_left,
                       df["income"].nunique()],
}, index=["Rows", "Columns", "Missing values (cells)", "Rows with missing values",
          "Exact duplicate rows", "Income labels"])

display(summary)
print(f"Columns after: 13 original + 2 engineered (age_group, has_capital_gain) = {df.shape[1]}")
print(f"Rows kept: {df.shape[0] / raw.shape[0] * 100:.1f}% of the original data")

## B8. Challenges (critical discussion)

**1. Missing values were hidden as `?`.**
*Problem:* The files use `?` with a leading space for missing values, so pandas read them as normal text and `.isna()` reported zero missing values (section B3.1). If this had gone unnoticed, `?` would have become a fake category in `workclass`, `occupation` and `native_country`, and would later have been encoded as if it were a real job type.
*Solution:* `skipinitialspace=True` removed the spaces, and `?` was replaced with NaN.
*Effect:* The real extent of missing data (about 7% of rows) became visible, which made an informed decision possible.

**2. Choosing how to handle missing values.**
*Problem:* Dropping rows loses data, but filling them adds guesses. Section B3.4 showed the gaps are **not random**: `workclass` and `occupation` are mostly missing together, and these rows contain fewer high earners, so they probably describe people outside regular work.
*Solution:* `dropna()` was chosen because the columns are categorical and the loss is small, while `"Unknown"` or mode filling would distort the categories.
*Effect:* The data is complete and still large, but it may slightly under-represent people who were not working. This is a known limitation of the results.

**3. Inconsistent target labels.**
*Problem:* The test file writes `>50K.` with a full stop, so the target had 4 labels instead of 2. A model would treat `>50K` and `>50K.` as different classes, and about a third of the data would be labelled wrongly.
*Solution:* `.str.rstrip(".")` removed the full stop before anything else was done.
*Effect:* One consistent target, which was then converted to 0/1.

**4. The order of cleaning steps changed the result.**
*Problem:* `fnlwgt` makes almost every row unique. Once it is removed, many *different* people share the same age, job, hours and so on, and would look like duplicates (section B4.2 shows how many).
*Solution:* Exact duplicates were removed **before** dropping `fnlwgt`, so only true copies were deleted.
*Effect:* Many genuine records were kept that a careless order would have deleted (the exact number is shown in section B4.2).

**5. The IQR rule was misleading for skewed columns.**
*Problem:* Most `capital_gain` and `capital_loss` values are 0, so the IQR is 0 and every non-zero value is called an outlier. The value 99,999 is a census cap, not a measurement.
*Solution:* Outliers were judged by meaning, not only by the formula: they are all valid values, so they were kept, and a simpler `has_capital_gain` feature was added.
*Effect:* High earners stay in the data, but the skew remains, so scaling will be needed before PCA.

**Wider limitation:** The data is from **1994** and covers only the **USA**. Incomes, jobs and social patterns have changed since then, and results may not apply to other countries such as Pakistan today. The target is also **imbalanced** (only about a quarter earn >50K), which must be kept in mind when evaluating models in later tasks.

## B9. Save and download the data

In [ ]:
# Save the cleaned dataset for the next tasks
df.to_csv("adult_clean.csv", index=False)
print("Saved adult_clean.csv with shape", df.shape)

In [ ]:
# Download the original data
from google.colab import files
files.download("adult_raw.csv")

In [ ]:
# Download the cleaned data
from google.colab import files
files.download("adult_clean.csv")

# Part C: Exploratory Data Analysis (Task 2)
This part looks for **variation** (how one column is spread), **covariation** (how two columns move together) and **unusual patterns**, and answers four questions about **income inequality** with charts.

## C0. Load the clean data

In [ ]:
# --- Libraries for the EDA ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")          # clean background with grid lines
plt.rcParams["figure.dpi"] = 100          # sharper charts

# Load the cleaned data saved at the end of Part B
df = pd.read_csv("adult_clean.csv")

# After reading a CSV, age_group is plain text. Give it its natural order again.
age_order = ["17-25", "26-35", "36-45", "46-55", "56-65", "66+"]
df["age_group"] = pd.Categorical(df["age_group"], categories=age_order, ordered=True)

# education_num 1-16 -> readable names (used for chart labels)
edu_names = {1: "Preschool", 2: "1st-4th", 3: "5th-6th", 4: "7th-8th", 5: "9th",
             6: "10th", 7: "11th", 8: "12th", 9: "HS-grad", 10: "Some-college",
             11: "Assoc-voc", 12: "Assoc-acdm", 13: "Bachelors", 14: "Masters",
             15: "Prof-school", 16: "Doctorate"}

print("Shape:", df.shape)
df.head()

## C1. Variation: one column at a time
### C1.1 Numeric columns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Age: histogram with a smooth density line (kde)
sns.histplot(df["age"], bins=30, kde=True, ax=axes[0], color="steelblue")
axes[0].set(title="Distribution of age", xlabel="Age (years)", ylabel="Number of people")

# Hours per week
sns.histplot(df["hours_per_week"], bins=30, ax=axes[1], color="seagreen")
axes[1].set(title="Distribution of hours worked per week", xlabel="Hours per week", ylabel="Number of people")

# Education level (discrete 1-16, so one bar per level)
sns.histplot(df["education_num"], discrete=True, ax=axes[2], color="darkorange")
axes[2].set(title="Distribution of education level", xlabel="Education level (1 = Preschool, 16 = Doctorate)",
            ylabel="Number of people")

plt.tight_layout(); plt.show()

# Figures used in the interpretation
print(f"Age: median {df['age'].median():.0f}, skewness {df['age'].skew():.2f}, "
      f"share aged 25-45: {df['age'].between(25, 45).mean()*100:.1f}%")
print(f"Hours: exactly 40 hours = {(df['hours_per_week'] == 40).mean()*100:.1f}% of people")
top_edu = df["education_num"].map(edu_names).value_counts(normalize=True).head(3).mul(100).round(1)
print("Most common education levels (%):", top_edu.to_dict())

**Interpretation:**
- **Age** is **right-skewed** (positive skewness): most people are aged about 25–45, and the number falls steadily after 50.
- **Hours per week** has a very tall, narrow peak at **40 hours** (the standard full-time week). Variation around it is small, with a few people working very short or very long weeks.
- **Education** is concentrated at three levels: **HS-grad**, **Some-college** and **Bachelors**. Very few people have a doctorate or less than 9th-grade education.

### C1.2 Categorical columns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# order=... sorts bars from most to least common
for ax, col, title in zip(axes.flat,
                          ["sex", "race", "workclass", "marital_status"],
                          ["Sex", "Race", "Type of employer (workclass)", "Marital status"]):
    sns.countplot(data=df, y=col, order=df[col].value_counts().index, ax=ax, color="steelblue")
    ax.set(title=f"Number of people by {title.lower()}", xlabel="Number of people", ylabel=title)

plt.tight_layout(); plt.show()

# Shares (%) used in the interpretation
for col in ["sex", "race", "workclass", "marital_status"]:
    print(col, df[col].value_counts(normalize=True).mul(100).round(1).head(3).to_dict())

**Interpretation:**
- **Sex:** men outnumber women by about **2 to 1**, so the sample is not balanced by sex.
- **Race:** the large majority of people are **White**; other groups are small, so results for them are less reliable.
- **Workclass:** most people work in the **Private** sector, far ahead of self-employment and government jobs.
- **Marital status:** **Married-civ-spouse** (married, living with a spouse) and **Never-married** are the two largest groups.

### C1.3 Income (the target)

In [ ]:
ax = sns.countplot(data=df, x="income", color="steelblue")
ax.set(title="Number of people by income group", xlabel="Income", ylabel="Number of people")
ax.set_xticks([0, 1], ["<=50K", ">50K"])

# Write the percentage on top of each bar
shares = df["income"].value_counts(normalize=True).sort_index() * 100
for i, s in enumerate(shares):
    ax.text(i, ax.patches[i].get_height(), f"{s:.1f}%", ha="center", va="bottom", fontsize=12)
plt.show()

**Interpretation:** the classes are **imbalanced**: about **three quarters** of people earn <=50K and about **one quarter** earn >50K. A model that always guessed "<=50K" would already be right about 75% of the time, so accuracy alone will be misleading in later tasks.

## C2. Covariation: two columns together
### C2.1 Correlation heatmap (numeric columns)

In [ ]:
corr = df.corr(numeric_only=True)      # Pearson correlation between every pair of numeric columns

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation between numeric columns")
plt.tight_layout(); plt.show()

# Correlation of each column with income, strongest first
print("Correlation with income:")
print(corr["income"].drop("income").sort_values(ascending=False).round(3))

# Show the order of correlation with income as text
from IPython.display import Markdown, display
order = corr["income"].drop("income").sort_values(ascending=False)
display(Markdown("**Order of correlation with income (strongest first):** " +
                 ", ".join(f"`{k}` ({v:.2f})" for k, v in order.items())))

**Interpretation:**
- No numeric column has a **strong** correlation with income on its own. The exact order is printed under the heatmap. **education_num**, **age**, **hours_per_week** and the capital-gain columns all have **positive** correlations: more education, more experience and longer hours go with higher income.
- **capital_gain** and **has_capital_gain** are linked because one was created from the other (Part B6), so only one of them should be used together in a model.
- The numeric columns are only weakly related **to each other**, which means each one brings different information. This matters for PCA in Task 3.
- Correlation only measures **straight-line** relationships and cannot include categorical columns such as sex or marital status, so these are explored with the questions below.

In [ ]:
# Covariation between a numeric and the target: age by income group
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="income", y="age", hue="income", palette="Set2", legend=False)
plt.xticks([0, 1], ["<=50K", ">50K"])
plt.title("Age by income group"); plt.xlabel("Income"); plt.ylabel("Age (years)")
plt.show()
print(df.groupby("income")["age"].median().rename({0: "<=50K", 1: ">50K"}))

**Interpretation:** people earning >50K are clearly **older** on average (higher median age), and the young group under 25 is almost entirely <=50K. Income rises with age and experience.

## C3. Supporting questions
Each question is about **income inequality**, the problem chosen in Part A.

### Q1. Does more education mean higher income?

In [ ]:
# % of people earning >50K at each education level
edu_rate = df.groupby("education_num")["income"].mean().mul(100)
edu_rate.index = edu_rate.index.map(edu_names)       # numbers -> names

ax = edu_rate.plot(kind="bar", figsize=(12, 5), color="steelblue")
ax.axhline(df["income"].mean() * 100, color="red", ls="--", label="Average of all people")
ax.set(title="Q1. Share earning >50K by education level", xlabel="Education level",
       ylabel="% earning >50K")
ax.legend(handles=[ax.lines[0]], loc="upper left"); plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

print(edu_rate.round(1)[["HS-grad", "Bachelors", "Masters", "Doctorate"]])

**Answer: Yes.** The share of high earners rises almost steadily with education. It is very low for people who did not finish high school, stays below average for **HS-grad** and **Some-college**, and rises sharply from **Bachelors** upwards. More than **70%** of people with a **Doctorate** or **Prof-school** degree earn >50K, compared with under 20% of high-school graduates. Education is one of the strongest single drivers of income in this data.

### Q2. Is there an income gap between men and women?

In [ ]:
# Row percentages: within each sex, what share earns <=50K and >50K?
sex_tab = pd.crosstab(df["sex"], df["income"], normalize="index").mul(100)
sex_tab.columns = ["<=50K", ">50K"]

ax = sex_tab.plot(kind="bar", stacked=True, figsize=(7, 5), color=["lightgray", "steelblue"])
ax.set(title="Q2. Income group by sex", xlabel="Sex", ylabel="% of people")
plt.xticks(rotation=0); plt.legend(title="Income", bbox_to_anchor=(1, 1)); plt.tight_layout(); plt.show()

print(sex_tab.round(1))
ratio = sex_tab.loc["Male", ">50K"] / sex_tab.loc["Female", ">50K"]
print(f"Men are {ratio:.1f} times as likely as women to earn >50K")

**Answer: Yes, there is a large gap.** Men are roughly **three times** as likely as women to earn >50K (about 3 in 10 men compared with about 1 in 10 women). Part of the gap may come from differences in hours worked, jobs and marital roles (see C4), but the size of the difference shows a clear inequality by sex.

### Q3. Do people who work more hours earn more?

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="income", y="hours_per_week", hue="income", palette="Set2", legend=False)
plt.xticks([0, 1], ["<=50K", ">50K"])
plt.title("Q3. Hours worked per week by income group")
plt.xlabel("Income"); plt.ylabel("Hours per week")
plt.show()

print(df.groupby("income")["hours_per_week"].describe()[["mean", "25%", "50%", "75%"]]
        .rename(index={0: "<=50K", 1: ">50K"}).round(1))

# Share of >50K among people working more vs less than 40 hours
print("\n% earning >50K:")
print(df.groupby(pd.cut(df["hours_per_week"], [0, 39, 40, 99],
                        labels=["Under 40", "Exactly 40", "Over 40"]),
                 observed=True)["income"].mean().mul(100).round(1))

**Answer: Yes, but only partly.** People earning >50K work **more hours** on average, and the middle 50% of their box sits higher. The share of high earners is much bigger among people working **over 40 hours** than among part-time workers. However, both groups have the **same typical value of 40 hours**, and the boxes overlap a lot, so hours alone do not explain income: many people work long hours and still earn <=50K.

### Q4. Does marital status affect income?

In [ ]:
mar_rate = df.groupby("marital_status")["income"].mean().mul(100).sort_values()

ax = mar_rate.plot(kind="barh", figsize=(9, 5), color="steelblue")
ax.axvline(df["income"].mean() * 100, color="red", ls="--", label="Average of all people")
ax.set(title="Q4. Share earning >50K by marital status", xlabel="% earning >50K", ylabel="Marital status")
ax.legend(handles=[ax.lines[0]], loc="lower right"); plt.tight_layout(); plt.show()

print(pd.DataFrame({"% >50K": mar_rate.round(1),
                    "people": df["marital_status"].value_counts()}).sort_values("% >50K"))

**Answer: Yes, strongly.** People who are **married and living with a spouse** (Married-civ-spouse) are far more likely to earn >50K than any other group, while **Never-married** people have the lowest share. This does not mean marriage *causes* higher income: married people are also **older** (see C2) and, in this data, are mostly **men** (see C4). The *Married-AF-spouse* group (armed-forces spouse) is very small, so its figure is not reliable.

## C4. Unusual patterns
The code below backs up each pattern with numbers.

In [ ]:
# 1 and 2: census cap values
print("capital_gain = 99,999:", (df["capital_gain"] == 99999).sum(), "people")
print("  next-highest value:", df.loc[df["capital_gain"] < 99999, "capital_gain"].max())
print("hours_per_week = 99:", (df["hours_per_week"] == 99).sum(), "people")
print("  people at 98 hours:", (df["hours_per_week"] == 98).sum())

In [ ]:
# 3: marriage, relationship and sex are tied together
print("Relationship of married-civ-spouse people (%):")
print(df.loc[df["marital_status"] == "Married-civ-spouse", "relationship"]
        .value_counts(normalize=True).mul(100).round(1))

print("\n% earning >50K by relationship:")
print(df.groupby("relationship")["income"].mean().mul(100).round(1).sort_values(ascending=False))

# Heatmap: sex vs relationship (counts)
plt.figure(figsize=(8, 3))
sns.heatmap(pd.crosstab(df["sex"], df["relationship"]), annot=True, fmt="d", cmap="Blues")
plt.title("Sex by relationship (number of people)"); plt.xlabel("Relationship"); plt.ylabel("Sex")
plt.tight_layout(); plt.show()

# Records that do not match: a Female "Husband" or a Male "Wife"
odd = df[((df["relationship"] == "Husband") & (df["sex"] == "Female")) |
         ((df["relationship"] == "Wife") & (df["sex"] == "Male"))]
print("Female 'Husband' or Male 'Wife' records:", len(odd))
if len(odd):
    display(odd[["age", "sex", "relationship", "marital_status", "income"]])

In [ ]:
# 4 and 5: imbalance in sex and income
print("Men per woman:", round((df["sex"] == "Male").sum() / (df["sex"] == "Female").sum(), 2))
print("Income split (%):", df["income"].value_counts(normalize=True).mul(100).round(1)
      .rename({0: "<=50K", 1: ">50K"}).to_dict())

**Unusual patterns found:**
1. **`capital_gain = 99,999`** appears for a group of people while no one has values just below it (there is a large jump from the next-highest value). This is a **census cap**: it means "very high gain", not an exact amount.
2. **`hours_per_week = 99`** has far more people than 98 or 97. It is also a **cap** for "99 or more hours", so long working weeks are bunched at one value.
3. **Marital status, relationship and sex overlap.** Almost all *Married-civ-spouse* people are recorded as **Husband** or **Wife**, and *Husband* is male in almost every record. *Husband* and *Wife* have the highest share of high earners. So `marital_status`, `relationship` and `sex` carry much of the **same information**, and part of the "marriage effect" in Q4 is really an effect of sex and age.
4. **Men outnumber women about 2 to 1.** Results for women are based on a smaller group, and the sample does not mirror the real population.
5. **Income is imbalanced (about 75/25).** Later models need to be judged with measures beyond accuracy (for example precision and recall).
6. **Sex and relationship do not always agree.** The code above lists any records of a **Female "Husband"** or a **Male "Wife"**. Only 2 such records were found, and they look like **data-entry errors**. They are too few to affect the results, so they were kept, but they show that census answers were not fully checked.

## C5. Summary of findings

In [ ]:
# Key findings, all calculated from the data
from IPython.display import Markdown, display
s = df.groupby("sex")["income"].mean() * 100
e = df.groupby("education_num")["income"].mean() * 100
m = df.groupby("marital_status")["income"].mean() * 100
h = df.groupby("income")["hours_per_week"].mean()
top = corr["income"].drop("income").sort_values(ascending=False)
odd_n = len(odd)

bullets = f"""**Main findings (exact figures from the data):**
- **Education** is the strongest single factor: **{e[16]:.1f}%** of Doctorate holders earn >50K, compared with **{e[9]:.1f}%** of high-school graduates. Its correlation with income is **{top['education_num']:.2f}**.
- **Gender gap:** **{s['Male']:.1f}%** of men earn >50K, compared with **{s['Female']:.1f}%** of women (**{s['Male']/s['Female']:.1f} times** as likely).
- **Hours:** people earning >50K work **{h[1]:.1f}** hours a week on average, compared with **{h[0]:.1f}** for <=50K, but both groups typically work 40 hours.
- **Marital status:** **{m['Married-civ-spouse']:.1f}%** of married people earn >50K, compared with **{m['Never-married']:.1f}%** of never-married people. This is mixed up with sex and age, because most married people in the data are husbands.
- **Data issues:** only **{df['income'].mean()*100:.1f}%** earn >50K (imbalanced), there are **{(df['sex']=='Male').sum()/(df['sex']=='Female').sum():.1f}** men per woman, and cap values exist (**{(df['capital_gain']==99999).sum()}** people at capital_gain = 99,999; **{(df['hours_per_week']==99).sum()}** at 99 hours)."""
display(Markdown(bullets))